<font size="6">Модели на основе энкодера Трансформера</font>

Архитектура классического Трансформера состоит из энкодера и декодера. Она используется для задачи машинного перевода — преобразования одной последовательности в другую, длина которых может не совпадать (sequence-to-sequence).

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/transformer.png" width="400"></center>

<center><em>Источник: <a href="https://arxiv.org/abs/1706.03762">Attention Is All You Need</a></em></center>

Однако блоки энкодера и декодера можно использовать по отдельности.
- Модели на основе декодера применяются для генерации текста и используют маскированное внимание (Generative Pre-trained Transformers, GPT)
- Модели на основе энкодера применяются для других задач: классификации одного или пары предложений, теггирования последовательности, поиска ответа на вопрос (Bidirectional Encoder Representations from Transformers, BERT)

Сегодня мы подробно рассмотрим архитектуру модели BERT и её применение для различных задач. BERT возник как результат исправления недочетов предыдущих моделей, поэтому рассказ про него мы начнем немного издалека.

## Первая модель на улице Сезам — ELMo

Модель ELMo была представлена в статье [Deep contextualized word representations](https://arxiv.org/abs/1802.05365).

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/elmo.png" width="600"></center>

<center><em>Источник: <a href="https://arxiv.org/abs/1802.05365">Deep contextualized word representations</a></em></center>

Различным значениям слова *play* соответствуют разные контексты
употребления. Нужно передавать не только значение слова, но и контекстуальную
информацию – **контекстуализированные векторные
представления слов**. Контекстуализированные эмбеддинги присваивают словам разные векторы на основе их семантики в контексте предложения. Такие контекстуализированные векторы вычисляются посредством обучения языковой модели: Embeddings from Language Models = ELMo.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/elmo_2.png" width="800"></center>

<center><em>Источник: <a href="https://www.google.com/url?sa=i&url=https%3A%2F%2Fx.com%2Fcatherinehyeo%2Fstatus%2F1283883310519705600%3Flang%3Dar-x-fm&psig=AOvVaw1P-Ip7QomOk7i0co0sy1kV&ust=1735482361749000&source=images&cd=vfe&opi=89978449&ved=0CBcQjhxqFwoTCPCFh-DVyooDFQAAAAAdAAAAABAE">This trend started with ELMo</a></em></center>

Для обучения векторов ELMo используется двунаправленная языковая модель (bidirectional Language Model или biLM).

Модель вычисляет вероятность последовательности $t_1, t_2, \dots, t_N$.
Два прохода по тексту:
- прямой (forward):
  - информация об определенном слове и контексте перед ним
  - вероятность $t_k$ при условии предшествующего контекста $t_1, ..., t_{k-1}$
- обратный (backward):
  - информация о слове и контексте после него
  - вероятность $t_k$ при условии последующего контекста $t_{k+1}, \dots, t_N$

Важно: ELMo не имеет отношения к Трансформерам.

Модель состоит из двух слоев. На каждом слое обучается двунаправленная модель долгой краткосрочной памяти (biLSTM).

Информация из прямого и обратного прохода на первом слое формирует промежуточные векторы слов, которые подаются на вход второго слоя модели. Результирующие векторы — взвешенная сумма необработанных векторов и двух промежуточных векторов.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/elmo_architecture.png" width="600"></center>

<center><em>Источник: <a href="https://www.researchgate.net/publication/359157231_Identifying_Contradictions_in_the_Legal_Proceedings_Using_Natural_Language_Models">Identifying Contradictions in the Legal Proceedings Using Natural Language Models</a></em></center>

Поскольку модель обучается на задаче языкового моделирования, размеченные тексты не нужны, появляется возможность использовать большой объем данных для обучения. Модель выучивает некоторые общие знания о языке, не затачиваясь ни под какую конкретную задачу.

ELMo стала важным шагом к распространению переноса обучения в области NLP. Выходы модели ELMo могут использоваться как контекстуализированные эмбеддинги для различных задач обработки текста.

В случае word2vec каждому слову соответствует конкретный вектор, они могут быть сохранены в файл и затем взязы оттуда.

ELMo строит контекстно зависимые вектора. Чтобы получить вектор для слов предложения, нужно сначала пропустить это предложение через модель. Обучения уже не происходит.

Векторы для слова *bank*: значения 'берег' и 'финансовая организация'.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bank_embeddings.png" width="600"></center>

<center><em>Источник: <a href="https://www.google.com/url?sa=i&url=https%3A%2F%2Ftowardsdatascience.com%2Fvisualizing-elmo-contextual-vectors-94168768fdaa&psig=AOvVaw2Fl8nHlIT3AsSv7fABstha&ust=1735482707884000&source=images&cd=vfe&opi=89978449&ved=0CBcQjhxqFwoTCOCok4TXyooDFQAAAAAdAAAAABAE">Visualizing ELMo Contextual Vectors</a></em></center>

## Вторая модель на улице Сезам — BERT

Модель BERT была представлена в статье [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805).

Идеи, которые были предложены ранее и удачно объединились при создании модели BERT:
- обучение на задаче языкового моделирования, которая не требует разметки, и перенос обучения (или предобучение) — ELMo
- механизм множественного внутреннего внимания, используемый без RNN, — Трансформер
- декодер Трансформера, который использует только левый контекст входного предложения, — GPT

Недостаток ELMo: анализируя левый и правый контекст отдельно с помощью biLSTM, мы можем терять часть информации. Хотелось бы учитывать левый и правый контекст одновременно.

Новшество BERT — использование **энкодера** Трансформера, чтобы получить "обогащенные" вниманием векторы слов.


<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bert.png" width="600"></center>

<center><em>Источник: <a href="https://www.linkedin.com/posts/robert-mcmenemy-%F0%9F%91%BE-70a0709b_efficient-language-modelling-with-bert-leveraging-activity-7247876446411452417-Fqzk">Efficient Language Modeling with BERT</a></em></center>

BERT состоит из нескольких последовательно соединенных блоков энкодера трансформера.

На вход модель получает последовательность токенов, на выходе отдает векторное представление для каждого токена, обогащенное контекстом. Энкодер содержит механизм внутреннего внимания (Self-Attention), который применяется к каждому токену и позволяет улавливать контекст.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bert_input.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

Две конфигурации:
- базовая (base): 12 слоев, размер скрытого слоя — 768, 110 миллионов весов
- расширенная (large): 24 слоя, размер скрытого слоя — 1024, 340 миллионов весов

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/bert_base_large.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

BERT обучается на двух задачах:
- маскированное языковое моделирование (masked language modeling, MLM)
- предсказание следующего предложения (next sentence prediction, NSP)

### Маскированное языковое моделирование

15% случайно выбранных токенов по всему корпусу маскируется — заменяется на спецтокен [MASK]. Задача модели — предсказать наиболее вероятный токен на месте маски.

Если модель всегда должна будет предсказывать наиболее вероятное слово только для масок, то для остальных слов векторы будут обучаться хуже. Нужно "обмануть" модель, чтобы она смотрела на все слова входной последовательности.

Среди выбранных 15% токенов:
- 80% маскируются: my dog is [MASK]
- 10% меняются на случайное слово: my dog is apple
- 10% остаются: my dog is hairy

Это разбиение меняется на каждой эпохе обучения.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/masked_language_modeling.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

Обучение на задаче маскированного языкового моделирования позволяет получить контекстуализированные векторы токенов. Предобученные векторы можно использовать в других задачах обработки текста.

### Предсказание следующего предложения

Помимо векторов слов, хотелось бы получать также векторные представления предложений. Для этого попробуем предсказать, следует ли одно предложение за другим.

Предложения разделены спецтокеном [SEP]. За классификацию отвечает спецтокен [CLS]. Он содержит представление обо всем предложении. Выход [CLS] токена пропускается через линейный слой размера 2.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/next_sentence_prediction.png" width="800"></center>

<center><em>Источник: <a href="https://jalammar.github.io/illustrated-bert/">The Illustrated BERT</a></em></center>

Положительные примеры представляют собой предложения, которые действительно следуют друг за другом в корпусе.

Вход: [CLS] the man went to the [MASK] store [SEP] he bought a gallon [MASK] milk [SEP]

Метка: IsNext

Отрицательные примеры представляют собой предложения, которые выбираются случайно.

Вход: [CLS] the man [MASK] to the store [SEP] penguin [MASK] are flight ##less birds [SEP]

Метка: NotNext

Обучение происходит по двум задачам параллельно. Значение функции потерь считается отдельно для маскированного языкового моделирования по токену [MASK] и для предсказания следующего слова по токену [CLS].

## Библиотека Transformers

Библиотека [Transformers 🛠️[doc]](https://huggingface.co/docs/transformers/index) создана сообществом HuggingFace — это платформа и сообщество для разработки и обмена моделями машинного обучения в области обработки естественного языка (и не только). Здесь можно найти готовые модели, узнать об их параметрах и применении, а также делиться своими разработками и идеями с другими специалистами. Библиотека [Transformers 🛠️[doc]](https://huggingface.co/docs/transformers/index) позволяет работать с открытыми трансформерными моделями.

In [ ]:
!pip install transformers -q

В библиотеке реализованы классы для различных архитектур — в том числе, для модели [BERT 🛠️[doc]](https://huggingface.co/docs/transformers/model_doc/bert).

Нам понадобятся классы [BertTokenizer 🛠️[doc]](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertTokenizer) и [BertForMaskedLM 🛠️[doc]](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertForMaskedLM). Для загрузки конкретных моделей используется метод `.from_pretrained`.

BERT — это общее название архитектуры. С её использованием были обучены модели на различных языках и датасетах. Все модели, которые доступны на HuggingFace, можно посмотреть в разделе [Models 🛠️[doc]](https://huggingface.co/models). Чтобы загрузить модель, нужно указать её идентификатор.

### BertTokenizer

Загрузим токенизатор для модели [BERT base cased 🛠️[doc]](https://huggingface.co/bert-base-cased) для английского языка.

In [ ]:
from transformers import BertTokenizer
en_tz = BertTokenizer.from_pretrained("google-bert/bert-base-cased")

Токенизируем английское предложение с помощью метода `.tokenize()`.

In [ ]:
sent = "He remains characteristically confident and optimistic."
tokenized_sent = en_tz.tokenize(sent)
tokenized_sent

Если какое-то слово не представлено в словаре целиком, при токенизации оно делится на подслова.

Посмотрим, какие индексы в словаре соответствуют словам, с помощью метода `convert_tokens_to_ids()`.

In [ ]:
en_tz.convert_tokens_to_ids(tokenized_sent)

Загрузим токенизатор для модели на основе архитектуры BERT для другого языка (не английского) и попробуем подобрать предложение, где при токенизации одно или более слов делятся на подслова.

Пример для русского языка:

In [ ]:
ru_tz = BertTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

In [ ]:
ru_tz.tokenize("Сельскохозяйственно-машиностроительный"), ru_tz.tokenize("Частнопредпринимательский")

Пример для немецкого языка:

In [ ]:
de_tz = BertTokenizer.from_pretrained("google-bert/bert-base-german-cased")

In [ ]:
de_tz.tokenize("die Aufmerksamkeitsdefizitstörung"), de_tz.tokenize("das Ampelmännchen")

### BertForMaskedLM

Загрузим саму модель [BERT base cased 🛠️[doc]](https://huggingface.co/bert-base-cased) для английского языка.

In [ ]:
from transformers import BertForMaskedLM
en_model = BertForMaskedLM.from_pretrained("google-bert/bert-base-cased")

Поскольку модель обучалась на задаче маскированного языкового моделирования, она способна предсказывать наиболее вероятные слова на месте спецтокена [MASK].

Напишем функцию `predict_mask`, которая находит распределение вероятностей для маски.
- Добавим спецтокены [CLS] и [SEP]
- Токенизируем текст (`text`)
- Определим индекс маскированного слова
- Переведем токенизированные слова (`tokenized_text`) в индексы
- Запишем индексы в тензор
- Применим модель к токенизированному предложению
- Запишем выходы модели для каждого слова
- Применим софтмакс (`torch.softmax()`) к результатам для маскированного слова, его найдем среди всех выходов модели (`predictions`) по индексу (`masked_index`)
- Запишем k самых больших значений весов и их индексы, которые соответствуют словам в словаре
- Пройдем в цикле по списку индексов
  - Переведем каждый индекс в соответствующий токен
  - Запишем его вероятность

In [ ]:
import torch

def predict_mask(tokenizer, model, text, top_k=5):

    text = f"[CLS] {text} [SEP]"
    tokenized_text = tokenizer.tokenize(text)
    masked_index = tokenized_text.index("[MASK]")
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])

    with torch.no_grad():
        outputs = model(tokens_tensor)
        predictions = outputs["logits"].squeeze() # token_len x vocabulary_size

    probs = torch.softmax(predictions[masked_index,:], dim=-1)
    top_k_weights, top_k_indices = torch.topk(probs, top_k)

    for i, pred_idx in enumerate(top_k_indices):
        predicted_token = tokenizer.convert_ids_to_tokens([pred_idx])[0]
        token_weight = top_k_weights[i]
        print("[MASK]: '%s'"%predicted_token, " | weights:", float(token_weight))

In [ ]:
predict_mask(en_tz, en_model, "My [MASK] is so cute.", top_k=5)

Загрузим модель на основе архитектуры BERT для другого языка (не английского) и применим функцию к предложению, где одно слово маскировано.

Пример для русского языка:

In [ ]:
ru_model = BertForMaskedLM.from_pretrained("DeepPavlov/rubert-base-cased")

In [ ]:
predict_mask(ru_tz, ru_model, "Моя [MASK] очень милая.", top_k=5)

Пример для английского языка:

In [ ]:
de_model = BertForMaskedLM.from_pretrained("google-bert/bert-base-german-cased")

In [ ]:
predict_mask(de_tz, de_model, "Meine [MASK] ist sehr nett.", top_k=5)

## Другие модели-энкодеры и их сравнение

После появления модели BERT стали появляться другие модели на основе энкодера трансформера.

**Модель RoBERTa**

[[paper] 🎓 Liu Y. et al. (2019). RoBERTa: A Robustly Optimized BERT Pretraining Approach](https://arxiv.org/abs/1907.11692)

Является улучшенной версией модели BERT за счет более тщательного подбора гиперпараметров.
- больший объем обучающих данных (в 10 раз больше — от 16 Гб к 160 Гб);
- увеличен размер батча (от 256 к 8 000) и размер словаря (от 30 000 к 50 000);
- динамическое маскирование: для каждого предложения используется 10 разных способов маскирования, через каждые 4 прохода по последовательности меняется позиция токена, который заменяется на маску;
- обучается только для маскированного языкового моделирования, предсказание следующего предложения исключается.


**Модель ALBERT**

[[paper] 🎓 Lan Zh. et al. (2020). ALBERT: A Lite BERT for Self-supervised Learning of Language Representations](https://arxiv.org/abs/1909.11942)

Сокращает количество параметров по сравнению с моделью BERT без снижения качества.
- факторизованная параметризация эмбеддинга

Для модели BERT E = H (E — размер эмбеддингов, H — размер скрытого слоя). Слой эмбеддингов имеет размер V x E (V — размер словаря). При увеличении размера скрытого слоя  увеличивается размер слоя эмбеддингов. Для BERTbase E = H = 768, для BERTlarge E = H = 1024

Чтобы размер скрытого слоя и размерность эмбеддинга были разными, после слоя эмбеддингов (V x E) добавляется полносвязный слой (E x H). Позволяет увеличить размер скрытого слоя, не меняя фактического размера эмбеддинга. Для модели ALBERT E = 128, H = 4096.
 - обмен параметрами между слоями

Архитектуры сетей на основе Трансформера полагаются на независимость слоев. Однако было замечено, что нейросеть выучивается выполнять схожие операции на разных слоях. Эта возможная избыточность устраняется в ALBERT с помощью обмена параметрами между слоями множественного внимания и полносвязными слоями.

- определение порядка предложений

Вместо предсказания следующего предложения используется новая задача: определение порядка предложений (Sentence Order Prediction, SOP).  Положительные примеры — два последовательных предложения, отрицательные примеры — те же, но с их измененным порядком.

<center><img src ="https://edunet.kea.su/repo/EduNet_NLP-web_dependencies/L06/BERT_fine_tuning.png" width="600"></center>

<center><em>Источник: <a href="https://arxiv.org/abs/1810.04805">BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding</a></em></center>

Как же сравнивать качество работы моделей?

Модели на основе энкодера используются для того, чтобы получать контекстуализированные векторные представления слов и предложений. Далее предобученные модели применяются для решения различных задач:

- классификация пары предложений
  - [Multi-Genre Natural Language Inference (MultiNLI) 🛠️[doc]](https://cims.nyu.edu/~sbowman/multinli/) — определение логической связи между текстами (для двух утверждений A и B выяснить, следует ли B из A)
  - [Microsoft Research Paraphrase Corpus (MRPC) 🛠️[doc]](https://www.microsoft.com/en-us/download/details.aspx?id=52398) — определить, являются ли предложения парафразами (выражают одинаковый смысл)
- классификация одного предложения
  - [Corpus of Linguistic Acceptability (CoLA) 🛠️[doc]](https://nyu-mll.github.io/CoLA/) — бинарная классификация предложений по приемлемости (приемлемые, неприемлемые)
  - [Stanford Sentiment Treebank (SST) 🛠️[doc]](https://nlp.stanford.edu/sentiment/index.html) — бинарная классификация по тональности (позитивные, негативные)
- поиск ответа на вопрос
  - [Stanford Question Answering Dataset (SQuAD) 🛠️[doc]](https://rajpurkar.github.io/SQuAD-explorer/) — выделить в тексте подпоследовательность, которая является ответом на заданный вопрос
- теггирование последовательности
  - [CoNLL-2003 🛠️[doc]](https://aclanthology.org/W03-0419.pdf) — распознавание именованных сущностей (имен людей, названий организаций, топонимов и т.п.)

Наборы для тестирования моделей представлены в бенчмарках [GLUE 🛠️[doc]](https://gluebenchmark.com/) и [SuperGLUE 🛠️[doc]](https://super.gluebenchmark.com/) для английского языка. Для оценки русскоязычных моделей существует бенчмарк [Russian SuperGLUE 🛠️[doc]](https://russiansuperglue.com/).

# Тонкая настройка BERT

Библиотека Transformers от Hugging Face позволяет скачивать предобученные модели и использовать их как начальный блок для подсчета контекстных векторов слов. Поверх этого блока добавляются другие слои, их архитектура зависит от задачи. Веса трансформерных моделей предобучены, но мы проводим тонкую настройку (fine-tuning) или дообучение на целевой задаче.

Мы рассмотрим, как можно осуществлять дообучение модели BERT для классификации предложений. Для этого мы изучим:

- Как подготовить датасет
- Как использовать высокоуровневое API для дообучения модели
- Как использовать собственный цикл обучения (training loop)

Установите библиотеки [Transformers 🛠️[doc]](https://huggingface.co/docs/transformers/index), [Datasets 🛠️[doc]](https://huggingface.co/docs/datasets/index) и [Evaluate 🛠️[doc]](https://huggingface.co/docs/evaluate/index).

In [ ]:
!pip install datasets evaluate transformers -q

## Предобработка данных

В данном разделе мы будем использовать датасет [Microsoft Research Paraphrase Corpus (MRPC) 🛠️[doc]](https://www.microsoft.com/en-us/download/details.aspx?id=52398). Он состоит из 5801 пар предложений с соответствующим им лейблом: является ли пара преложений парафразами или нет (т.е. идет ли речь в обоих предложениях об одном и том же). Мы выбрали именно этот датасет, потому что он небольшой: с ним легко экспериментировать в процессе обучения.

### Загрузка датасета

[Hugging Face 🛠️[doc]](https://huggingface.co/) содержит не только модели, там также расположено множество [датасетов 🛠️[doc]](https://huggingface.co/datasets) для различных языков и задач.

Но сейчас вернемся к датасету MRPC! Это один из 10 датасетов бенчмарка [GLUE 🛠️[doc]](https://gluebenchmark.com/) для оценки моделей машинного обучения в задачах классификации текста.

Мы можем загрузить датасет следующим образом:

In [ ]:
from datasets import load_dataset

raw_mrpc = load_dataset("glue", "mrpc")
raw_mrpc

Как можно заметить, мы получили объект типа `DatasetDict`, который включает обучающую выборку, валидационную выборку и тестовую выборку. Каждая из них содержит несколько колонок (`sentence1`, `sentence2`, `label` и `idx`) и переменную с числом элементов (`num_rows`): 3668 пар предложений в обучающей части, 408 в валидационной и 1725 в тестовой .

Мы можем получить доступ к предложениями в объекте `raw_datasets` путем индексирования, как в словаре:

In [ ]:
raw_train_mrpc = raw_mrpc["train"]
print("First element in train dataset:\n")
raw_train_mrpc[0]

Можно увидеть, что лейблы уже являются целыми числами (integer), их обрабатывать не нужно. Чтобы сопоставить индекс класса с его названием, можно вывести значение переменной `features` у `raw_train_dataset`:

In [ ]:
print("Feature types:\n")
raw_train_mrpc.features

Переменная `label` типа `ClassLabel` соответствует именам в `names`. `0` соответствует `not_equivalent`, `1` соответствует `equivalent`.

### Предобработка датасета

Чтобы предобработать датасет, нам необходимо конвертировать текст в числа, которые может обработать модель. Как мы видели ранее, это делается с помощью токенизатора. Мы можем подать на вход токенизатору одно предложение или список, т.е. можно токенизировать предложения попарно таким образом:

In [ ]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenized_sentences_1 = tokenizer(raw_mrpc["train"]["sentence1"])
tokenized_sentences_2 = tokenizer(raw_mrpc["train"]["sentence2"])

Однако мы не можем просто передать две последовательности в модель и получить прогноз того, являются ли эти два предложения парафразами или нет. Нам нужно обрабатывать две последовательности как пару и применять соответствующую предварительную обработку. К счастью, токенизатор также может взять пару последовательностей и подготовить их так, как ожидает наша модель BERT:

In [ ]:
sentence1 = "This is the first sentence."
sentence2 = "This is the second one."
inputs = tokenizer(sentence1, sentence2)
print(f"First sentence: {sentence1}")
print(f"Second sentence: {sentence2}")
print("Result of tokeniization:")
inputs

`input_ids` содержит индексы, соответствующие токенам по словарю.

Маски внимания (`attention_mask`) — это тензоры той же формы, что и тензор входных идентификаторов, заполненные 0 и 1: 1 означает, что соответствующие токены должны “привлекать внимание”, а 0 означает, что соответствующие токены не должны “привлекать внимание” (т.е. должны игнорироваться слоями внимания модели).

`token_type_ids` указывает модели, какая часть входных данных является первым предложением, а какая вторым.

Если мы декодируем ID из `input_ids` обратно в слова, мы получим:

In [ ]:
print("Converting indices to tokens:")
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

Токенизатор уже содержит индексы для спецсимволов:
- [SEP] — метка конца предложения
- [CLS] — токен для классификации предложения
- [PAD] — токен для выравнивания длин последовательностей

Видно, что модель ожидает входные данные в следующем формате: `[CLS] sentence1 [SEP] sentence2 [SEP]` в случае двух предложений. Посмотрим соответствие элементов и `token_type_ids`.

In [ ]:
for token, id in zip(tokenizer.convert_ids_to_tokens(inputs["input_ids"]), inputs["token_type_ids"]):
  print(token, id)

Как вы можете заметить, части входных данных, соответствующих `[CLS] sentence1 [SEP]` имеют тип токена `0`, в то время как остальные части, соответствующие второму предложению `sentence2 [SEP]`, имеют тип токена `1`.

Обратите внимание, что если вы выберете другой идентификатор модели, `token_type_ids` необязательно будут присутствовать в ваших токенизированных входных данных. Они возвращаются только тогда, когда модель будет знать, что с ними делать, потому что она видела их во время предобучения.

В данном случае BERT был обучен с информацией об идентификаторах типов токенов, и помимо задачи маскированного языкового моделирования, он может решать еще одну задачу: предсказание следующего предложения (next sentence prediction). Суть этой задачи — смоделировать связь между предложениями.

В этой задаче модели на вход подаются пары предложений (со случайно замаскированными токенами), от модели требуется предсказать, является ли следующее предложение продолжением текущего. Чтобы задача не была слишком тривиальной, половина времени модель обучается на соседних предложениях из одного документа, другую половину на парах предложений, взятых из разных источников.

В общем случае вам не нужно беспокоиться о наличии `token_type_ids` в ваших токенизированных данных: пока вы используете одинаковый чекпоинт и для токенизатора, и для модели – токенизатор будет знать, как нужно обработать данные.

Теперь мы знаем, что токенизатор может подготовить сразу пару предложений, а значит мы можем использовать его для целого датасета: можно подать на вход токенизатору список первых предложений и список вторых предложений. Это также сработает и для механизмов дополнения (padding) и усечения до максимальной длины (truncation).

Итак, один из способов предобработать обучающий датасет:

In [ ]:
tokenized_mrpc = tokenizer(
    raw_mrpc["train"]["sentence1"],
    raw_mrpc["train"]["sentence2"],
    padding=True,
    truncation=True,
)

У данного способа есть недостаток: токенизатор возвращает объект с ключами, `input_ids`, `attention_mask`, и `token_type_ids` и значениями в формате списка списков. Это будет работать только если у нас достаточно оперативной памяти (RAM) для хранения целого датасета во время токенизации.

In [ ]:
for k, v in tokenized_mrpc.items():
  print(f"Key: {k}, value type: {type(v)}")

Чтобы хранить данные в формате датасета, мы будем использовать методы `Dataset.map()`. Метод `map()` применяет некоторую функцию к каждому элементу датасета.

Давайте определим функцию, которая токенизирует наши входные данные:

In [ ]:
def tokenize_function_mrpc(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

Эта функция принимает на вход словарь (похожий на элементы нашего словаря) и возвращает новый словарь с ключами `input_ids`, `attention_mask` и `token_type_ids`. Заметьте, это также работает, если словарь `example` содержит несколько элементов (каждый ключ в виде списка предложений), поскольку `tokenizer` работает и со списками пар предложений, как мы и видели ранее. Это позволит нам использовать аргумент `batched=True` в вызове `map()`, который ускорит процесс токенизации.

Обратите внимание, мы оставили аргумент `padding` пустым, потому что дополнение данных до максимальной длины неэффективно: гораздо быстрее делать это во время формирования батча, в таком случае мы будем дополнять до максимальной длины только элементы батча, а не целого датасета. Это поможет сэкономить время в случае длинных последовательностей.

Ниже пример того, как мы применяем функцию токенизации к целому датасету. Поскольку мы указываем `batched=True` при вызове `map`, функция будет применена сразу к нескольким элементам датасета одновременно, а не к каждому по отдельности. Это позволяет сделать токенизацию более быстрой.

In [ ]:
tokenized_mrpc = raw_mrpc.map(tokenize_function_mrpc, batched=True)
tokenized_mrpc

### Dynamic padding

Функция, отвечающая за объединение элементов внутри батча, называется `collate_function`. Это аргумент, который вы можете передать при создании `DataLoader`. По умолчанию это функция, которая просто преобразует объекты в тензоры PyTorch и объединяет их. В нашем случае это невозможно, поскольку входные данные, которые у нас есть, не будут иметь одинакового размера. Мы намеренно не стали делать паддинг, чтобы применять его только по мере необходимости в каждом батче и избегать слишком длинных входных данных с большим количеством отступов.

Функция `collate_function` будет осуществлять корректный паддинг элементов выборки, которые мы хотим объединить в батч. Библиотека [Transformers 🛠️[doc]](https://huggingface.co/docs/transformers/index) предоставляет эту функцию через класс `DataCollatorWithPadding`. При создании экземпляра требуется указать токенизатор: чтобы знать, какой токен использовать для паддинга и слева или справа нужно дополнять данные.

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Чтобы протестировать, давайте возьмем несколько элементов обучающей выборки, которые мы хотим объединить в батч. Мы удалим колонки `idx`, `sentence1` и `sentence2`, т.к. они содержат строки (а мы не можем превратить строки в тензоры) и посмотрим на длину каждой записи в батче:

In [ ]:
samples = tokenized_mrpc["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

Неудивительно: мы получили объекты разной длины от 32 до 67. Динамический паддинг подразумевает, что все объекты будут дополнены до максимальной длины в батче, то есть до 67.

Без динамического паддинга все предложения должны быть дополнены до максимальной длины во всем наборе данных, или до максимальной длины, которую может принять модель.

Давайте проверим, что `data_collator` динамически правильно дополняет батч:

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

Выглядит неплохо! Теперь мы пришли от обычного текста к батчу, с которым может работать наша модель.

Можем приступить к fine-tuning!